## EDA esercizio
<img src="https://frenzy86.s3.eu-west-2.amazonaws.com/python/EDA2.jpg" width=800 >


# TIPS

In [1]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

tips = pd.read_csv('https://frenzy86.s3.eu-west-2.amazonaws.com/fav/tips.csv')
tips

,total_bill,tip,sex,smoker,day,time,size
0,16.99,1.01,Female,No,Sun,Dinner,2
1,10.34,1.66,Male,No,Sun,Dinner,3
2,21.01,3.50,Male,No,Sun,Dinner,3
3,23.68,3.31,Male,No,Sun,Dinner,2
4,24.59,3.61,Female,No,Sun,Dinner,4
...,...,...,...,...,...,...,...
239,29.03,5.92,Male,No,Sat,Dinner,3
240,27.18,2.00,Female,Yes,Sat,Dinner,2
241,22.67,2.00,Male,Yes,Sat,Dinner,2
242,17.82,1.75,Male,No,Sat,Dinner,2


A data frame with 244 observations on the following 8 variables.


**total_bill:** a numeric vector, the bill amount (dollars)

**tip:**a numeric vector, the tip amount (dollars)

**sex:** factor with levels Female Male, gender of the payer of the bill

**smoker:** factor with levels No Yes, whether the party included smokers

**day:** factor with levels Friday Saturday Sunday Thursday, day of the week

**time:** factor with levels Day Night, rough time of day

**size:** numeric vector, number of people in party

In [2]:
# sns.set()
# tips = sns.load_dataset("tips")
# tips.head()

In [3]:
tips.describe().T

,count,mean,std,min,25%,50%,75%,max
total_bill,244.0,19.785943,8.902412,3.07,13.3475,17.795,24.1275,50.81
tip,244.0,2.998279,1.383638,1.00,2.0000,2.900,3.5625,10.00
size,244.0,2.569672,0.951100,1.00,2.0000,2.000,3.0000,6.00


In [4]:
tips.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 244 entries, 0 to 243
Data columns (total 7 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   total_bill  244 non-null    float64
 1   tip         244 non-null    float64
 2   sex         244 non-null    object 
 3   smoker      244 non-null    object 
 4   day         244 non-null    object 
 5   time        244 non-null    object 
 6   size        244 non-null    int64  
dtypes: float64(2), int64(1), object(4)
memory usage: 13.5+ KB


In [5]:
from sklearn.preprocessing import StandardScaler

#standardizzo 'total_bill','size' che sono le features, non posso standardizzare tip perchè è la Target
ss = StandardScaler()
tips[['total_bill','size']] = ss.fit_transform(tips[['total_bill','size']])
tips

,total_bill,tip,sex,smoker,day,time,size
0,-0.314711,1.01,Female,No,Sun,Dinner,-0.600193
1,-1.063235,1.66,Male,No,Sun,Dinner,0.453383
2,0.137780,3.50,Male,No,Sun,Dinner,0.453383
3,0.438315,3.31,Male,No,Sun,Dinner,-0.600193
4,0.540745,3.61,Female,No,Sun,Dinner,1.506958
...,...,...,...,...,...,...,...
239,1.040511,5.92,Male,No,Sat,Dinner,0.453383
240,0.832275,2.00,Female,Yes,Sat,Dinner,-0.600193
241,0.324630,2.00,Male,Yes,Sat,Dinner,-0.600193
242,-0.221287,1.75,Male,No,Sat,Dinner,-0.600193


In [6]:
X = tips[['total_bill','size']]
y = tips['tip']

In [7]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y,
                                                    test_size=0.25,
                                                    random_state=667,
                                                    )

In [8]:
from sklearn.linear_model import LinearRegression

model = LinearRegression()


In [9]:
model.fit(X_train,y_train)

LinearRegression()

In [10]:
y_pred = model.predict(X_test)

In [11]:
## 5 -  Misurare l'errore del mio modello
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error,root_mean_squared_error

mae = mean_absolute_error(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)
rmse = root_mean_squared_error(y_test, y_pred)
r2score = r2_score(y_test, y_pred)

ad_r2score = 1-(1-r2score)*(len(X_test)-1)/(len(X_test)-X_test.shape[1]-1)

print('MAE: ', mae)
print('MSE: ', mse)
print('RMSE: ', rmse)
print('R2_score: ', r2score)
print('Adjusted_R2_score: ', ad_r2score)

MAE:  0.7113992237969078
MSE:  1.1057306474641262
RMSE:  1.0515372782094443
R2_score:  0.46027832770597243
Adjusted_R2_score:  0.44166723555790244


In [12]:
import joblib

In [13]:
joblib.dump(model,'model_tips_stand.pkl')

['model_tips_stand.pkl']

In [14]:
## errore enorme!!!!!!!!!!!!!!!!!!!!!!!!!!
#loaded_model.predict([[40,3]])[0]  # l'input di un modello standardizzato deve essere standardizzato

In [15]:
#model_inferce(40,3)

In [ ]:
import gradio as gr
import joblib
loaded_model = joblib.load('model_tips_stand.pkl')


def model_inferce(Total_Bill,Size):
    bill_st = (Total_Bill -19.78)/8.90
    size_st = (Size -2.56)/0.95
    y_pred = loaded_model.predict([[bill_st,size_st]])[0]
    return y_pred.round(2)


title = "Calcola la TIP yo! "
description = "This application calcolate the tip"

demo = gr.Interface(
                    fn=model_inferce,
                    inputs=["number", "number"],
                    outputs="number",
                    title=title,
                    description=description,
                    flagging_mode= "never",
                    )

demo.launch(share=True,debug=True)